# Documentación de Mecanismos — Repositorio `bdbroldan`
**Fecha:** 28/05/2026  
**Autor:** BROLDAN  
**Alcance:** Mecanismo de Normalización (Consolidación) + Mecanismo de Pago de Mora

## Índice
1. [Estructura de carpetas](#estructura)
2. [Función Genérica — `funDataGenerica`](#generica)
3. [Mecanismo CONSOLIDACIÓN](#consolidacion)
   - 3.1 [HTML del FUN — `funconsolidacion.html`](#html-fun)
   - 3.2 [Creación de obligaciones sin BD — `procrearobl.js`](#crear-obl)
   - 3.3 [Carga de datos al FUN — `datafunconsolidacion.js`](#data-fun)
   - 3.4 [Botón para agregar obligaciones — `btncrearobl.js`](#btn-crear-obl)
   - 3.5 [Versión anterior de eventos — `eventofc.js`](#evento-fc)
4. [Mecanismo PAGO DE MORA](#mora)
   - 4.1 [Cálculo inicial — `calculomora.js`](#calculo-mora)
   - 4.2 [Recálculo por cambio de pago — `recalculo mora.js`](#recalculo)
   - 4.3 [Generación SOX — `soxmorapro.js`](#sox)
   - 4.4 [Notas de refactor](#notas-refactor)
5. [Mapa de IDs de campo (UUIDs)](#ids)
6. [Flujos de datos](#flujos)
7. [Bugs conocidos y pendientes](#bugs)

---
## 1. Estructura de carpetas <a id='estructura'></a>

```
bdbroldan/
└── bdbroldan/
    ├── CONSOLIDACION/
    │   ├── funconsolidacion.html       ← Plantilla HTML del FUN (modal + PDF)
    │   ├── btncrearobl.js              ← Botón "adicionar obligación": SweetAlert2 con contador ±
    │   ├── procrearobl.js              ← Versión activa: cards de obligación manual con contadorGlobal
    │   ├── crearoblsinbd.js            ← Versión anterior de creación de cards (sin contador global)
    │   ├── datafunconsolidacion.js     ← Versión refactorizada: lectura limpia de datos del formulario
    │   └── eventofc.js                 ← Versión anterior del cargador de datos (tiene bug saldoTotalDesembolsar)
    ├── FUN/
    │   └── data generica.js            ← Función común a todos los mecanismos: fecha, cliente, oficina
    └── MECANISMO MORA/
        ├── calculomora.js              ← Cálculo inicial al seleccionar obligación en PAGOMORA
        ├── recalculo mora.js           ← Recálculo al cambiar el valor de pago SNR
        ├── soxmora.js                  ← Versión anterior SOX (descuento mora hardcodeado 100%)
        ├── soxmorapro.js               ← Versión producción SOX (descuento mora dinámico)
        ├── REFACTOR CALCULOS MORA.txt  ← Notas técnicas del refactor (race conditions, bugs)
        ├── Calculos Mora/              ← Carpeta vacía (reservada)
        └── txt/                        ← Carpeta vacía (reservada)
```

---
## 2. Función Genérica — `funDataGenerica` <a id='generica'></a>

**Archivo:** `FUN/data generica.js`  
**Reutilizada en:** todos los mecanismos (consolidación, novación, mora, etc.)

### Propósito
Cargar los datos comunes del encabezado del FUN: fecha actual, datos de oficina y datos del cliente.

### Parámetro de entrada
| Parámetro | Tipo | Descripción |
|-----------|------|-------------|
| `mecanismo` | `string` | Sufijo identificador del mecanismo (`'consolidacion'`, `'novacion'`, etc.) |

### Datos que captura
| Campo interno | ID Lappiz | Descripción |
|---------------|-----------|-------------|
| `codOficina` | `9552efdb-f91c-4e51-9f55-230282926b12` | Código de oficina (select) |
| `nombreOficina` | `bd198dd5-328d-4ded-a3b4-b23adfad423a` | Nombre de oficina (input) |
| `pagoProduto` | `685c5e9d-4409-4d4c-a11e-a0c17dcedb02` | Cuenta SNR (solo novación/consolidación; el resto usa `000-778175`) |
| `nombreAsesor` | `sessionStorage.LappizUser.FullName` | Nombre del asesor reordenado (Apellido, Nombre → Nombre Apellido) |
| `nombreCliente` | `1ad60ed2-e515-4164-8270-54efa1e574fa` | Nombre del cliente |
| `tipoDoc` | `15fb0de1-4989-4986-a662-61fb88b3aba1` | Tipo de documento (limpia prefijo numérico `XX:`) |
| `numDoc` | `75fda36b-9317-4062-93d7-26d45e6188d6` | Número de documento |
| `lugarExp` | `d8faf1c6-44fb-4bd9-93ca-aac3c9ef6ab3` | Lugar de expedición |
| `fechaExp` | `64b5c3a4-2d66-4e35-97af-8a9405f0cf63` | Fecha de expedición (formato con espacios) |
| `diasMora` | `sessionStorage.EdadMoraCl` | Días de mora del cliente |

### Comportamiento notable
- Se ejecuta con `setTimeout(..., 500)` para garantizar que Lappiz haya pintado el DOM.
- El nombre del asesor viene en formato `"Apellidos, Nombres"` desde `LappizUser`; la función lo invierte.

---
## 3. Mecanismo CONSOLIDACIÓN <a id='consolidacion'></a>

La **Consolidación** es un mecanismo de normalización mediante el cual se unifican múltiples obligaciones en mora de un mismo cliente en un solo crédito con nuevas condiciones (plazo, cuota, tasa). Se genera el **FUN** (Formato Único de Normalización) como documento soporte.

### 3.1 HTML del FUN — `funconsolidacion.html` <a id='html-fun'></a>

**Rol:** Plantilla visual del documento oficial. Se renderiza dentro de un `<dialog id="modalconsolidacion">` y se exporta como PDF con `html2pdf.js`.

#### Secciones del documento

| # | Sección | IDs relevantes |
|---|---------|----------------|
| 1 | **Encabezado** | `day_consolidacion`, `month_consolidacion`, `year_consolidacion` |
| 2 | **Oficina** | `codOficina_consolidacion`, `nombreOficina_consolidacion`, `cuentaPago_consolidacion`, `funcionario_consolidacion` |
| 3 | **Información cliente** | `nombreCliente_consolidacion`, `tipoDoc_consolidacion`, `numDoc_consolidacion`, `lugarExp_consolidacion`, `fechaExp_consolidacion` |
| 4 | **Actividad e ingresos** | `diasMora_consolidacion`, `descripcionActividad_consolidacion`, `ingresoMensual_consolidacion`, `ocupacionAdicional_consolidacion`, `ingresosAdicionales_consolidacion` |
| 5 | **Tabla de obligaciones** | `filas-obligaciones-consolidacion` (contenedor dinámico; filas `numObligacion_N_consolidacion`, `saldoTotal_N_consolidacion`, etc.) |
| 6 | **Condiciones baja en cuenta** | `pctBenIntCorr_consolidacion`, `pctBenIntMora_consolidacion`, `pctBenIntExtra_consolidacion`, `nuevoPlazo_consolidacion`, `saldoTotal_consolidacion`, `valCuotaProyectada_consolidacion`, `pagoNegociacionNew_consolidacion`, `amortizacion_consolidacion`, `tasaIntEA_consolidacion` |
| 7 | **Aceptación / Términos** | Texto fijo legal. Inserta `nombreClienteTerd_consolidacion`, `numDocTerd_consolidacion` |
| 8 | **Preguntas FATCA** | `pregunta1_consolidacion` … `pregunta4_consolidacion` |
| 9 | **Observaciones y Garantías** | `observaciones_consolidacion`, `garantiaFAG_consolidacion`, `garantiaFNG_consolidacion` |
| 10 | **Firma** | Espacio visual, sin ID dinámico |

#### Notas de estilos
- Ancho del contenedor: 780 px (modal 850 px).
- Toda la tipografía usa escala `8–12 px` para imitar el formato físico del banco.
- La clase `.layout-especial_con` aplica el reset CSS dentro del scope del componente (evita colisión con Lappiz).

### 3.2 Creación de obligaciones sin BD — `procrearobl.js` <a id='crear-obl'></a>

**Función activa:** `obligacionSinBaseConsolidacion(cantidad)`  
**Versión anterior:** `crearoblsinbd.js` (mismo nombre de función pero sin `contadorGlobal`).

#### Cuándo se usa
Cuando el simulador **no tiene datos precargados desde BD** (modo ingreso manual). El asesor indica cuántas obligaciones quiere consolidar y la función genera N tarjetas de formulario.

#### Flujo interno
```
obligacionSinBaseConsolidacion(N)
  ├─ execQuery('EXEC SimiladorDNC_Lappiz_EmailConfirmed @sw = 11')
  │    └─ Devuelve lista de marcas de obligación (Id, Title, PeorMarca, MarcaLetra)
  └─ Por cada obligación 0..N-1:
       ├─ Usa contadorGlobal++ como índice único (evita duplicar IDs si se llama varias veces)
       ├─ Crea card (.card1) con:
       │    ├─ Input obligación (obligation-{index})
       │    ├─ Toggle switch (toggle-{index}) → habilita/deshabilita campos
       │    ├─ Inputs numéricos: saldoTotal, interesCorriente, interesMora, interesesExtracontables
       │    │    └─ Formato miles en tiempo real (formatNumber)
       │    └─ Select marca obligación (populated desde el execQuery)
       └─ Agrega la card a #consolidacion
```

#### Diferencia entre `crearoblsinbd.js` y `procrearobl.js`
| Aspecto | `crearoblsinbd.js` | `procrearobl.js` |
|---------|-------------------|------------------|
| Índice de IDs | `i` (relativo a la llamada) | `contadorGlobal++` (global, evita duplicados en llamadas múltiples) |
| `data-label` en obligation input | Sí | No (solo clase) |
| Estado | **Obsoleto** | **Activo** |

### 3.3 Carga de datos al FUN — `datafunconsolidacion.js` <a id='data-fun'></a>

**Función:** `DataFunConsolidacion(mecanismo)`  
**Versión:** Refactorizada (usa optional chaining y fallbacks seguros).

#### Responsabilidad
Leer los campos del formulario Lappiz, estructurarlos y escribirlos en los IDs del HTML del FUN.

#### Subfunciones internas

**`dataConsolidacion()`** — Lee y devuelve el objeto de datos de negocio:

| Campo devuelto | ID Lappiz | Descripción |
|----------------|-----------|-------------|
| `descripcionActividad` | `c852f2a7-...` | Actividad económica (select) |
| `ingresoMensual` | `67631aed-...` | Ingreso mensual (`aria-valuenow`) |
| `ocupacionAdicional` | `b54af750-...` | Ocupación adicional (select) |
| `ingresosAdicionales` | `1a47c2c1-...` | Ingresos adicionales |
| `totalBajaEnCuentaIntCte` | `04dbcb19-...` | % baja en cuenta interés corriente |
| `totalBajaEnCuentaIntMora` | `f848cad9-...` | % baja en cuenta interés mora |
| `totalBajaEnCuentaExtraContables` | `dc9166ce-...` | % baja en cuenta extracontables |
| `saldoTotalDesembolsar` | `69b7fc43-...` | Saldo total a desembolsar |
| `amortizacion` | `03011879-...` | Amortización |
| `plazo` | `aa4de771-...` | Nuevo plazo en meses |
| `tasaIntEA` | `c9f5317e-...` | Tasa de interés E.A. |
| `cuotaProyectada` | `e74b2587-...` | Valor cuota proyectada |
| `pagoNegociacion` | `0ee03528-...` | Pago para la gestión de recuperación |
| `observacionesPag4` | `be70a202-...` | Observaciones |
| `pregunta1`–`pregunta4` | IDs genéricos `pregunta1`–`pregunta4` | Preguntas FATCA |
| `garantiaFAG`, `garantiaFNG` | IDs genéricos | Garantías |

**`getObligacionesActivas()`** — Itera sobre todos los toggles activos en el DOM, extrae los valores numéricos (`data-raw-value`) de cada card y retorna un array de obligaciones:
```javascript
[
  { numObligacion, saldoTotal, intCorrientes, intMora, intExtraC, marcaObligacion },
  ...
]
```

**`llenarFilasObligaciones(obligaciones)`** — Escribe hasta 6 filas en el FUN usando el patrón `numObligacion_N_consolidacion`, `saldoTotal_N_consolidacion`, etc. Las filas sin dato se limpian.

**`loadFormData(data)`** — Escribe los campos de actividad e ingresos en el FUN.

#### Ejecución
```javascript
setTimeout(() => {
    const data = dataConsolidacion();
    const obligActivas = getObligacionesActivas();
    loadFormData(data);
    llenarFilasObligaciones(obligActivas);
}, 500);
```

### 3.4 Botón para agregar obligaciones — `btncrearobl.js` <a id='btn-crear-obl'></a>

**Evento:** `$(document).on('click', '#adicionar-obligacion', ...)`  
**Dependencias:** [SweetAlert2](https://sweetalert2.github.io/), `obligacionSinBaseConsolidacion` (de `procrearobl.js`)

#### Propósito
Proporcionar la interfaz de usuario para que el asesor indique cuántas tarjetas de obligación desea añadir al formulario. Actúa como el **punto de entrada** del flujo de creación manual de obligaciones.

#### Flujo del botón
```
Clic en #adicionar-obligacion
        ↓
Swal.fire() muestra diálogo con:
  ├─ Botón [-]  → decrementa contador (mínimo 1)
  ├─ Display    → muestra valor actual (inicia en 1)
  └─ Botón [+]  → incrementa contador (sin límite superior)
        ↓
Asesor confirma
        ↓
Lee cantidadFinal desde el elemento #cantidad del HTML del diálogo
        ↓
obligacionSinBaseConsolidacion(cantidadFinal)  ← genera N cards en #consolidacion
```

#### Detalles de implementación
| Aspecto | Detalle |
|---------|---------|
| Selector del botón | `#adicionar-obligacion` (delegación en `document`) |
| Modal | SweetAlert2 (`Swal.fire`) |
| Colores de botones ± | Amarillo `#ffe100` (−) / Azul BdB `#0041a4` (+) |
| Valor mínimo | 1 (el botón − no baja de 1) |
| Valor máximo | Sin límite (el botón + incrementa libremente) |
| Lectura del valor | `document.getElementById('cantidad').innerText` tras confirmar |
| Manejo de errores | `try/catch` con `console.error` |

#### Nota sobre la variable `cantidadFinal`
El valor se lee del DOM del diálogo **después** de que SweetAlert2 lo cierra. Aunque SweetAlert2 destruye el contenido del diálogo al cerrarse, el `.then()` se ejecuta de forma síncrona antes de la destrucción, por lo que `document.getElementById('cantidad')` aún existe en ese momento. Sin embargo, es un patrón frágil — si SweetAlert2 actualiza su ciclo de vida, podría romperse. Una alternativa más robusta sería guardar `valor` en la clausura del `didOpen` y pasarlo directamente al `.then()`.

### 3.5 Versión anterior — `eventofc.js` <a id='evento-fc'></a>

**Estado:** Obsoleto (no usar en producción).

#### Diferencias vs `datafunconsolidacion.js`
| Aspecto | `eventofc.js` | `datafunconsolidacion.js` |
|---------|---------------|---------------------------|
| IDs de baja en cuenta | `b42b41d8-...`, `e079d101-...`, `e970af6e-...` | `04dbcb19-...`, `f848cad9-...`, `dc9166ce-...` |
| Bug `saldoTotalDesembolsar` | Referencia variable sin declarar → error en runtime | Corregido con optional chaining |
| Manejo de `loadFormData` | Escribe directamente con `document.getElementById` | Usa subfunción `setCell` con guard |
| Sección obligaciones | No implementada | Implementada con `getObligacionesActivas` |

---
## 4. Mecanismo PAGO DE MORA <a id='mora'></a>

El mecanismo de **Pago de Mora** permite al cliente ponerse al día en una obligación en mora pagando el saldo vencido con descuentos parciales en intereses (corrientes, moratorios y extracontables para TC). No implica reestructuración del crédito.

### 4.1 Cálculo inicial — `calculomora.js` <a id='calculo-mora'></a>

**Función:** `CalculosMora()`  
**Trigger:** Se ejecuta al seleccionar una obligación en el grid de PAGOMORA (evento `onSelecting` o equivalente de Lappiz; usa `e.dataItem`).

#### Flujo
```
CalculosMora()
  1. Limpia sessionStorage (PorcPagoMoraIntCte1, PorcentajePagomora1, porcDescIntExtraCTC1, calculosMoraListo, reintentosRecalculo)
  2. Detecta si el producto es TARJETA
     ├─ TC: habilita campo intereses extracontables, guarda valor en sessionStorage
     └─ Cartera: fuerza todos los campos TC a 0 y los deshabilita
  3. Define colchon = TC ? 0 : 20.000 (buffer mínimo para cartera)
  4. Setea en el form: PagoMinObl, InteresCteObl, InteresMoraObl
  5. Ejecuta execQuery a SimiladorDNC_Lappiz_TasasVigentes
     WHERE RangoDias3 = {edadMora}
     → Obtiene PorcentajePagomora (% descuento mora) y PorcPagoMoraIntCte (% descuento corriente)
  6. Verifica si aplica campaña PAGOMORA (sobreescribe porcentajes con valores de campaña)
  7. Calcula:
     ├─ maxDescCTE   = porcDescIntCteIcs%  * InteresCteObl
     ├─ maxDescIntM  = porcDescIntMoraIcs% * InteresMoraObl
     └─ maxDescIntTC = porcDescIntExtraCTC% * InteresesExtraObl (solo TC)
  8. Setea máximos de descuento en el form
  9. Calcula abonoMinimo = PagoMinObl - MaxTotalDesc + colchon
 10. Guarda porcentajes en sessionStorage
 11. Marca calculosMoraListo = 'si'
```

#### Campos del formulario escritos
| Campo visual | ID Lappiz | Descripción |
|-------------|-----------|-------------|
| Es TC | `7a5c89e8-...` | `'Si'` / `'No'` (display) |
| Intereses extrac. TC | `aef7fd98-...` | Valor si es TC |
| Pago mínimo | `af9911f8-...` | De `e.dataItem.PagoMinObl` |
| Interés corriente | `9b3ac68c-...` | De `e.dataItem.InteresCteObl` |
| Interés mora | `c13b3910-...` | De `e.dataItem.InteresMoraObl` |
| Max descuento corriente | `36329717-...` | Calculado |
| Descuento corriente aplicado | `49ed37fa-...` | Calculado |
| % descuento corriente | `e076d650-...` | Calculado |
| Max descuento mora | `24a29872-...` | Calculado |
| Descuento mora aplicado | `db8c0e77-...` | Calculado |
| % descuento mora | `64fcdf9f-...` | Calculado |
| Max descuento extrac. | `de744073-...` | Solo TC |
| Descuento extrac. aplicado | `a01eeadb-...` | Solo TC |
| % descuento extrac. | `0456eeb3-...` | Solo TC |
| Total max descuentos | `6cfd4b2c-...` | Suma de los tres |
| Abono mínimo | `8f7266d7-...` | PagoMin - MaxDesc + colchon |
| Descuento total real | `6af98cad-...` | Idem MaxTotalDesc al inicio |

### 4.2 Recálculo por cambio de pago — `recalculo mora.js` <a id='recalculo'></a>

**Función:** `RecalculosMora()`  
**Trigger:** Se ejecuta cada vez que el asesor modifica el valor de pago SNR (`3539dba8-...`).

#### Propósito
Ajustar los descuentos reales en función del monto que el cliente efectivamente va a pagar. Si el cliente paga **más** del mínimo, se reducen los descuentos; si paga **menos**, se aumentan (modo excepción de negocio).

#### Guard de sincronización
```javascript
if (sessionStorage.calculosMoraListo !== 'si') {
    // Reintentar hasta 10 veces con 200ms de espera
    if (intentos < 10) { setTimeout(RecalculosMora, 200); return; }
}
```
Esto garantiza que `CalculosMora` haya terminado antes de recalcular.

#### Lógica de distribución de descuentos

```
excesoPago = max(0, PagoSNR - abonoMinimo)

PASO 1 — Reducir Corriente + ExtraC (proporcionalmente, simultáneo)
  Si exceso > 0:
    factor = (bolsa - exceso) / bolsa
    dctoCte   = maxCte * factor
    dctoExtraC = maxExtC * factor   (solo TC)
    exceso -= (maxCte - dctoCte) + (maxExtC - dctoExtraC)
  Si pago < abonoMinimo (déficit):
    factor = (bolsa + deficit) / bolsa
    dctoCte   = maxCte * factor     ← puede superar el máximo (excepción de negocio)
    dctoExtraC = maxExtC * factor

PASO 2 — Reducir Mora (solo si queda exceso)
  dctoMora -= min(dctoMora, exceso)
```

#### Invariante clave
- Los intereses **Corriente** y **ExtraContables (TC)** se mueven **siempre en la misma proporción**.
- El interés **Mora** solo se reduce después de que Corriente y ExtraC llegaron a cero.

### 4.3 Generación SOX — `soxmorapro.js` <a id='sox'></a>

**Función:** `soxMora()`  
**Versión activa:** `soxmorapro.js`  
**Versión anterior:** `soxmora.js` (descuento de mora hardcodeado al 100%)

#### Propósito
Generar automáticamente:
1. El texto de **observaciones** (campo narrativo de la negociación).
2. La cadena **SOX** (formato interno del banco para registrar la transacción).

#### Campos leídos
| Variable | ID Lappiz | Descripción |
|----------|-----------|-------------|
| `producto` | `caae86ca-...` (select) / `c5f3bb92-...` (texto) | Nombre del producto |
| `pagoMinimo` | `af9911f8-...` | Pago mínimo |
| `descIntMora` | `64fcdf9f-...` | % descuento mora (dinámico) |
| `descIntCte` | `e076d650-...` | % descuento corriente |
| `descTotal` | `6af98cad-...` | Descuento total aplicado |
| `pagoSNR` | `3539dba8-...` | Valor que paga el cliente |
| `excepcion` | `d3b8782c-...` | Excepción aplicada |
| `cuotaVencida` | `fc42583f-...` | Cuota vencida |
| `fechaPago` | `ee8b70aa-...` (input interno) | Fecha de pago (sin `/`) |

#### Diferencia entre versiones
| Aspecto | `soxmora.js` | `soxmorapro.js` |
|---------|-------------|----------------|
| % mora en texto | Hardcodeado `'100%'` | Dinámico: `${descIntMora}%` |
| Campo `descIntMora` | No leído | `64fcdf9f-...` |
| Salto de línea en SOX | No | Sí (hay `\n  ` en la cadena) |

#### Formato de la cadena SOX
```
FECHAPAGOXX{fecha}LLLVALORCONSIGSNRXX{pago}LLLVALORPAGOPRODUCTOXX0LLL
VALORHONORARIOSXX0LLLVALORPRODUCTOXX{pago}LLLTIPONEGXXTELLL
CUOTAPROYECTADAXXNO APLICALLLOBSERVACIONESXX{texto}LLL
EXCEPCIONXX{excepcion}LLLCUOTAVENCIDAXX{cuota}LLL
```

### 4.4 Notas del refactor de Calculos Mora <a id='notas-refactor'></a>

**Fuente:** `REFACTOR CALCULOS MORA.txt`

#### Cambios aplicados en `CalculosMora`
- **Limpieza total de `sessionStorage`** al inicio con `removeItem` (antes se sobreescribía, dejando valores rancios si el query fallaba).
- **`safeNumber` en todos los `e.dataItem`** para evitar explosión con `null` / `undefined`.
- **TC fuerza extracontables a 0** si el producto es cartera, sin importar lo que traiga la BD.
- **`sessionStorage.esTarjetaObl`** como fuente de verdad del tipo (el campo visual `7a5c89e8` es solo display).
- **`RecalculosMora()` se llama desde `.then()` y `.catch()`**, no con `setTimeout` (elimina la race condition documentada en T10).
- **`.catch()` explícito**: si el query falla, `sessionStorage` queda limpio y el formulario no queda zombie.

#### Cambios aplicados en `RecalculosMora`
- **Eliminado el `setTimeout`** que causaba la race condition.
- **Lee `sessionStorage.esTarjetaObl`** en lugar del campo visual.
- **Doble fuente para `InteresesExtraObl`**: primero campo visual (editable por el asesor), fallback a `sessionStorage`.
- **Bloque `else` explícito** para cartera: setea los campos de TC a 0 (antes quedaban residuales al alternar TC/cartera).

---
## 5. Mapa de IDs de campo (UUIDs) <a id='ids'></a>

### 5.1 Campos compartidos (todos los mecanismos)
| ID Lappiz | Campo |
|-----------|-------|
| `9552efdb-f91c-4e51-9f55-230282926b12` | Select código oficina |
| `bd198dd5-328d-4ded-a3b4-b23adfad423a` | Nombre oficina |
| `685c5e9d-4409-4d4c-a11e-a0c17dcedb02` | Cuenta SNR / Pago producto (novación + consolidación) |
| `1ad60ed2-e515-4164-8270-54efa1e574fa` | Nombre cliente |
| `15fb0de1-4989-4986-a662-61fb88b3aba1` | Tipo documento |
| `75fda36b-9317-4062-93d7-26d45e6188d6` | Número documento |
| `d8faf1c6-44fb-4bd9-93ca-aac3c9ef6ab3` | Lugar expedición |
| `64b5c3a4-2d66-4e35-97af-8a9405f0cf63` | Fecha expedición |

### 5.2 Campos exclusivos CONSOLIDACIÓN
| ID Lappiz | Campo |
|-----------|-------|
| `c852f2a7-6f9c-48f6-96b5-6fdc26c399ef` | Descripción actividad |
| `67631aed-75e4-4b23-8601-17cadd1c7003` | Ingreso mensual |
| `b54af750-167e-4831-bb8c-c374e7f45202` | Ocupación adicional |
| `1a47c2c1-4551-4d13-89ca-82e89ce655c0` | Ingresos adicionales |
| `04dbcb19-8f74-4eac-81f3-6bcc76cd7f9a` | % baja cta. int. corriente |
| `f848cad9-f94d-4e56-9468-863a2a55e402` | % baja cta. int. mora |
| `dc9166ce-a5c8-4fc7-ad2b-4c6479d63f12` | % baja cta. int. extracontables |
| `69b7fc43-675b-4984-bd64-9fd68799a97b` | Saldo total a desembolsar |
| `03011879-0560-4a41-826b-888c89f6ab83` | Amortización |
| `aa4de771-cbaf-486d-8de2-06941dc220d5` | Nuevo plazo (meses) |
| `c9f5317e-9099-43f1-9b7f-78b93d99aa6a` | Tasa interés E.A. |
| `e74b2587-dccc-4395-8333-f6c2f34338aa` | Cuota proyectada |
| `0ee03528-b018-47d1-856b-9e30dbae2ddf` | Pago negociación |
| `be70a202-71a9-40ea-851b-945702693b51` | Observaciones pág. 4 |

### 5.3 Campos exclusivos PAGO MORA
| ID Lappiz | Campo |
|-----------|-------|
| `5b9ce178-27fe-4c52-b91d-ba6a898ff546` | Dropdown edad mora (override manual) |
| `7a5c89e8-a431-4b76-b3bc-24f6a187978c` | Es TC (Si/No — display) |
| `caae86ca-b4e0-4e59-918e-8f7a1a4d4114` | Select nombre producto |
| `c5f3bb92-1efe-47ea-941a-5bf2c5f6ceb0` | Nombre producto (texto fallback) |
| `aef7fd98-0a00-4ec8-95d9-37840df1fe67` | Intereses extracontables TC |
| `af9911f8-4a06-4483-b25d-6bec9e1647fe` | Pago mínimo |
| `9b3ac68c-68ff-4928-864d-906e9d851621` | Interés corriente |
| `c13b3910-1960-422f-835d-7ea89982f8b6` | Interés mora |
| `3539dba8-0c22-491e-a05b-84642d675d59` | Pago SNR (ingresado por el asesor) |
| `36329717-6123-40c7-b4c9-d5f447a3cac4` | Max descuento corriente |
| `49ed37fa-10f7-46d1-b2d3-bd4e28bef0db` | Descuento corriente aplicado |
| `e076d650-c5d6-48b1-920b-295d431604b0` | % descuento corriente |
| `24a29872-6b5f-40fd-bae7-cb072e972ff5` | Max descuento mora |
| `db8c0e77-0029-4bf9-ba9a-ebc141721c33` | Descuento mora aplicado |
| `64fcdf9f-c6b3-4742-b4b2-e259759290d9` | % descuento mora |
| `de744073-f3bd-4c05-ac6f-9ca493664262` | Max descuento extracontables |
| `a01eeadb-b99e-4e08-9d93-3fe44b9e1cf8` | Descuento extracontables aplicado |
| `0456eeb3-8809-48a5-8726-87e416efdcb3` | % descuento extracontables |
| `6cfd4b2c-6ef4-4821-95d5-364657fda787` | Total max descuentos |
| `8f7266d7-dfc0-4ff4-afad-c50fbfa67062` | Abono mínimo |
| `6af98cad-1f96-4ad5-b33c-b0ddc8f68133` | Descuento total real aplicado |
| `d3b8782c-c94a-4b7a-a2aa-00baba7bfbd5` | Excepción |
| `fc42583f-067a-4bd6-9985-2962d447ad0f` | Cuota vencida |
| `ee8b70aa-2712-408c-a87a-b121e20564b3` | Fecha de pago |
| `96c93177-4705-4bd2-ac50-e304c007afa3` | Observaciones generadas |
| `b24357e4-d1be-443d-8fa0-5b8790a1c508` | Cadena SOX |

---
## 6. Flujos de datos <a id='flujos'></a>

### 6.1 Flujo CONSOLIDACIÓN (modo sin BD)

```
Asesor hace clic en #adicionar-obligacion
        ↓
btncrearobl.js → Swal.fire() con contador ± (mínimo 1)
        ↓
Asesor confirma cantidad N
        ↓
obligacionSinBaseConsolidacion(N)   ← procrearobl.js
  └─ execQuery(@sw=11) → marcas de obligación
  └─ Genera N cards en #consolidacion (índices con contadorGlobal)
        ↓
Asesor completa cada card (toggle ON + valores numéricos + marca)
        ↓
Asesor abre modal FUN → llama DataFunConsolidacion('consolidacion')
  ├─ funDataGenerica('consolidacion')   → encabezado, cliente, oficina
  ├─ dataConsolidacion()                → condiciones de negocio
  └─ getObligacionesActivas()           → obligaciones marcadas (hasta 6)
        ↓
FUN pintado en #modalconsolidacion
        ↓
Asesor exporta PDF con html2pdf.js
```

### 6.2 Flujo PAGO DE MORA

```
Asesor selecciona obligación en el grid
        ↓
CalculosMora()   ← evento onSelecting de Lappiz
  ├─ Detecta tipo producto (TC / Cartera)
  ├─ execQuery(SimiladorDNC_Lappiz_TasasVigentes) → porcentajes por rango días
  ├─ Aplica campaña si corresponde
  ├─ Calcula maxDesc* y abonoMinimo
  ├─ Setea campos en el form
  └─ sessionStorage.calculosMoraListo = 'si'
        ↓
Asesor ingresa valor de pago SNR
        ↓
RecalculosMora()   ← onChange del campo pago SNR
  ├─ Guard: espera calculosMoraListo
  ├─ Lee porcentajes desde sessionStorage
  ├─ Calcula distribución exceso/déficit
  └─ Actualiza descuentos reales en el form
        ↓
Asesor genera SOX
        ↓
soxMora()   ← botón o evento
  ├─ Lee todos los valores del form
  ├─ Genera texto de observaciones
  └─ Genera cadena SOX y la setea en el campo correspondiente
```

---
## 7. Bugs conocidos y pendientes <a id='bugs'></a>

### Bugs documentados en `REFACTOR CALCULOS MORA.txt`

| # | Escenario | Síntoma | Estado |
|---|-----------|---------|--------|
| 1 | **Sin data (modo manual) — extracontable** | Al ingresar un valor en el campo de intereses extracontables manualmente, el `%` de extracontable vuelve a cero → fallo en el cálculo del descuento | **Abierto** |
| 2 | **Con data (modo BD) — extracontable** | El valor de extracontables queda con el porcentaje de la base en lugar del que define la política (debe quedar igual que corriente) | **Abierto** |
| 3 | **Con data — mora > 120 días** | El % de mora está quedando en 50% cuando la política para > 120 días es 75% | **Abierto** |

### Bugs corregidos en el refactor

| # | Descripción | Archivo afectado |
|---|-------------|------------------|
| T10 | Race condition: `RecalculosMora` se ejecutaba antes que el `execQuery` de `CalculosMora` terminara | `calculomora.js` + `recalculo mora.js` |
| — | Valores rancios de obligación anterior en `sessionStorage` | `calculomora.js` |
| — | Campos TC con valores residuales al alternar producto | `recalculo mora.js` |
| — | `saldoTotalDesembolsar` sin declarar en `eventofc.js` | Resuelto usando `datafunconsolidacion.js` |
| — | SOX con % mora fijo 100% | `soxmora.js` → reemplazado por `soxmorapro.js` |